In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
#os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

In [3]:
from pinecone import Pinecone, ServerlessSpec

index_name = "hybrid-search-rag"
pc = Pinecone(api_key=pinecone_api_key)
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384, #Dimension of huggingface embedding
        metric='dotproduct', #sparse value onlyy support dot product
        spec=ServerlessSpec(cloud='aws',region="us-east-1")
    )

ImportError: cannot import name 'Pinecone' from 'pinecone' (unknown location)

In [7]:
index = pc.Index(index_name)
index

Index(host='https://hyybrid-search-rag-fbyyecv.svc.aped-4627-b74a.pinecone.io')

In [8]:
import os
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
embeddings

ModuleNotFoundError: No module named 'langchain_huggingface'

In [ ]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default() # defaulltt use TFIDF
bm25_encoder

In [ ]:
sentences=[
    "In 2023, I visiteed Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
]

bm25_encoder.fit(sentences)
bm25_encoder.dump("bm25_values.json")
bm25_encoder = BM25Encoder().load("bm25_values.json")

In [ ]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

retriever = PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)
retriever

In [ ]:
retriever.add_texts(
    [
    "In 2023, I visiteed Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans"
]
)

In [ ]:
retriever.invoke("What city did I visit recent?")
retriever.invoke("What city did I visit first?")